In [55]:
import numpy as np
from sklearn import preprocessing
import tensorflow as tf
raw_csv_data = np.loadtxt('Audiobooks_data.csv',delimiter=',')
unscaled_imputs_all = raw_csv_data[:,1:-1]
tagets_all = raw_csv_data[:,-1]

In [56]:
shuffled_indices = np.arange(unscaled_imputs_all.shape[0])
np.random.shuffle(shuffled_indices)
unscaled_imputs_all = unscaled_imputs_all[shuffled_indices]
tagets_all = tagets_all[shuffled_indices]

In [57]:
num_one_tagets = int(np.sum(tagets_all))
zero_tagets_counter =0
indices_to_remove = []

for i in range(tagets_all.shape[0]):
  if tagets_all[i] == 0:
    zero_tagets_counter += 1
    if zero_tagets_counter > num_one_tagets:
      indices_to_remove.append(i)

unscaled_inputs_equal_priors = np.delete(unscaled_imputs_all,indices_to_remove,axis=0)
tagets_equal_priors = np.delete(tagets_all,indices_to_remove,axis=0)

In [58]:
scaled_inputs = preprocessing.scale(unscaled_inputs_equal_priors)

In [59]:
shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)
shuffled_inputs = scaled_inputs[shuffled_indices]
shuffled_targets = tagets_equal_priors[shuffled_indices]

In [60]:
from sklearn.utils import validation
samples_count = shuffled_inputs.shape[0]

train_samples_count = int(0.8 * samples_count)
validation_samples_count = int(0.1 * samples_count)

test_samples_count = samples_count - train_samples_count - validation_samples_count

train_inputs = shuffled_inputs[:train_samples_count]
train_targets = shuffled_targets[:train_samples_count]

validation_inputs = shuffled_inputs[train_samples_count:train_samples_count+validation_samples_count]
validation_targets = shuffled_targets[train_samples_count:train_samples_count+validation_samples_count]

test_inputs = shuffled_inputs[train_samples_count+validation_samples_count:]
test_targets = shuffled_targets[train_samples_count+validation_samples_count:]

In [61]:
print(np.sum(train_targets), train_samples_count, np.sum(train_targets) / train_samples_count)
print(np.sum(validation_targets), validation_samples_count, np.sum(validation_targets) / validation_samples_count)
print(np.sum(test_targets), test_samples_count, np.sum(test_targets) / test_samples_count)

1792.0 3579 0.5006985191394244
215.0 447 0.4809843400447427
230.0 448 0.5133928571428571


In [63]:
np.savez('Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_data_test', inputs=test_inputs, targets=test_targets)

In [35]:
########################################################################

In [64]:

npz = np.load('Audiobooks_data_train.npz')


train_inputs = npz['inputs'].astype(np.float32)
train_targets = npz['targets'].astype(np.int32)

npz = np.load('Audiobooks_data_validation.npz')
validation_inputs, validation_targets = npz['inputs'].astype(np.float32), npz['targets'].astype(np.int32)


npz = np.load('Audiobooks_data_test.npz')
test_inputs, test_targets = npz['inputs'].astype(np.float32), npz['targets'].astype(np.int32)

In [65]:
input_size = 10
output_size = 2
hidden_layer_size = 200

model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='elu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='elu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='elu'),
    tf.keras.layers.Dense(output_size, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [66]:
batch_size = 32
max_epochs = 200

early_stopping = tf.keras.callbacks.EarlyStopping(
    patience=10,
    restore_best_weights=True
)

model.fit(
    train_inputs,
    train_targets,
    batch_size=batch_size,
    epochs=max_epochs,
    callbacks=[early_stopping],
    validation_data=(validation_inputs, validation_targets),
    verbose=1
)

Epoch 1/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7530 - loss: 0.4419 - val_accuracy: 0.7740 - val_loss: 0.4108
Epoch 2/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7614 - loss: 0.4185 - val_accuracy: 0.7763 - val_loss: 0.4016
Epoch 3/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7751 - loss: 0.4070 - val_accuracy: 0.7964 - val_loss: 0.3807
Epoch 4/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7748 - loss: 0.3955 - val_accuracy: 0.7248 - val_loss: 0.4065
Epoch 5/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7742 - loss: 0.3964 - val_accuracy: 0.7718 - val_loss: 0.3935
Epoch 6/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7935 - loss: 0.3833 - val_accuracy: 0.7785 - val_loss: 0.3910
Epoch 7/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7865 - loss: 0.3869 - val_accuracy: 0.7673 - val_loss: 0.3718
Epoch 8/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7840 - loss: 0.3837 - val_accu

In [68]:
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)
print('Test loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8192 - loss: 0.3584 
Test loss: 0.36. Test accuracy: 81.92%
